<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 3 — Exercises: Embeddings, Chunking & Vector Search

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Practise

Eight short tasks. Fill in every `___` and run the cell.

1. **Q1** — find the odd one out using similarity alone
2. **Q2** — plot your own embedding space
3. **Q3** — add a sentence that bridges two topics, and predict where it lands
4. **Q4** — hunt for the chunk size that destroys a fact
5. **Q5** — prove the recursive splitter fixes it
6. **Q6** — index a document of your own choosing
7. **Q7** — filter by metadata, and watch a filter force a bad answer
8. **Q8** — cite your sources in a RAG answer

> **Q1–Q7 need no API key at all.** Only Q8 does.

---

## 1. Setup

Run these three cells. Nothing to fill in yet — the first one takes a minute (it downloads the model).

In [ ]:
# PROVIDED - just run this cell.
!pip install -q chromadb sentence-transformers scikit-learn litellm langchain-text-splitters matplotlib

In [ ]:
# PROVIDED - just run this cell.
import os
import numpy as np
import matplotlib.pyplot as plt
from getpass import getpass
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

model = SentenceTransformer('all-MiniLM-L6-v2')   # 384 numbers, free, local
chroma_client = chromadb.Client()

print("Ready - no API key needed until Q8.")

---

## Q1: Odd one out

Four sentences. Three are about the same thing. Find the outlier **using similarity only** —
no keywords, no rules.

In [ ]:
# Hint: cosine_similarity(vectors) gives the full matrix.
#       .mean(axis=1) averages each row - the outlier has the LOWEST average.
#       .argmin() gives you the index of the lowest value.

sentences = [
    "The chef prepared a delicious curry",
    "This recipe needs fresh coriander",
    "We ate dinner at a new restaurant",
    "The train leaves from platform nine",
]

vectors = model.___(sentences)
matrix = ___(vectors)

averages = matrix.mean(axis=1)
odd_index = averages.___()

print("Average similarity per sentence:", averages.round(3))
print("Odd one out:", sentences[odd_index])

---

## Q2: Plot your own embedding space

Write **six sentences of your own** — three on one topic, three on another — and see whether
they separate. Pick any two topics you like.

In [ ]:
# Hint: PCA(n_components=2).fit_transform(vectors) turns 384 numbers into 2.
#       coords[:, 0] is the x column, coords[:, 1] is the y column.

my_sentences = [
    "___",   # topic A
    "___",   # topic A
    "___",   # topic A
    "___",   # topic B
    "___",   # topic B
    "___",   # topic B
]

my_vectors = model.encode(my_sentences)
coords = ___(n_components=___).fit_transform(my_vectors)

colors = ["#6366F1"] * 3 + ["#0891B2"] * 3

plt.figure(figsize=(9, 6))
plt.scatter(coords[:, 0], coords[:, ___], c=colors, s=150)
for (x, y), label in zip(coords, my_sentences):
    plt.annotate(label, (x, y), fontsize=9, xytext=(6, 4), textcoords="offset points")
plt.axis("off")
plt.show()

---

## Q3: The bridge sentence

Now write **one sentence that belongs to both of your topics at once**.

**Predict where it will land before you run the cell.** Say it out loud, then look.

In [ ]:
# Hint: a bridge mentions both topics, e.g. music + weather ->
#       "We played an outdoor concert in the rain".
#       Give it its own colour so you can spot it.

bridge = "___"

all_sentences = my_sentences + [bridge]
all_vectors = model.encode(all_sentences)
coords = PCA(n_components=2).fit_transform(all_vectors)

colors = ["#6366F1"] * 3 + ["#0891B2"] * 3 + ["___"]     # make the bridge red

plt.figure(figsize=(9, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=150)
for (x, y), label in zip(coords, all_sentences):
    plt.annotate(label, (x, y), fontsize=9, xytext=(6, 4), textcoords="offset points")
plt.axis("off")
plt.show()

---

## Q4: Find the chunk size that destroys a fact

The sentence below says the library is open for **9 hours**. Find a naive chunk size that
**splits that fact in half**, so no single chunk contains it.

In [ ]:
# Hint: "9 hours" starts at character 24. You need a cut that lands INSIDE it -
#       try something between 25 and 30.

FACT = "The library is open for 9 hours every weekday and closes at 8pm."


def naive_chunks(text, size):
    return [text[i:i + size] for i in range(0, len(text), size)]


size = ___
broken = naive_chunks(FACT, size)

print(broken)
print("\nIs '9 hours' in any chunk?", any("9 hours" in c for c in broken))   # want False

---

## Q5: Prove the recursive splitter fixes it

Same sentence, same rough chunk size — but a splitter that respects word boundaries.

In [ ]:
# Hint: RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=...)
#       Try a chunk_size around 40 with an overlap around 10.

splitter = ___(chunk_size=___, chunk_overlap=___)
good = splitter.split_text(FACT)

print(good)
print("\nIs '9 hours' in any chunk?", any("9 hours" in c for c in ___))      # want True

---

## Q6: Index a document of your own

Paste **two or three paragraphs** of anything — a Wikipedia article, your own notes, a product
page — then chunk it, embed it and store it.

In [ ]:
# Hint: splitter.split_text(text)  ->  model.encode(chunks).tolist()  ->  collection.add(...)
#       Chroma wants plain Python lists, not numpy arrays.

MY_TEXT = """
___
"""

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
my_chunks = doc_splitter.split_text(___)

my_col = chroma_client.get_or_create_collection(name="my_document")
my_col.add(
    ids=[f"c{i}" for i in range(len(my_chunks))],
    embeddings=model.encode(my_chunks).___(),
    documents=___,
)

print(my_col.count(), "chunks indexed")

# Now ask it something - using words that are NOT in your text if you can
answer = my_col.query(query_embeddings=model.encode(["___"]).tolist(), n_results=2)
print(answer["documents"][0])

---

## Q7: Metadata filter — and how a filter forces a bad answer

Store four campus facts, each tagged with a topic. Then run the **same query** three ways.

In [ ]:
# Hint: metadatas takes one dict per document.
#       col.query(..., where={"topic": "academics"}) filters BEFORE searching.

facts = [
    "The canteen serves lunch from 12pm to 2pm",
    "The library is open until 10pm",
    "Assignments are due every Friday",
    "The exam hall opens at 9am",
]
tags = [{"topic": "food"}, {"topic": "campus"}, {"topic": "academics"}, {"topic": "academics"}]

fcol = chroma_client.get_or_create_collection(name="campus_facts")
fcol.add(
    ids=["f0", "f1", "f2", "f3"],
    embeddings=model.encode(facts).tolist(),
    documents=facts,
    metadatas=___,
)

q = model.encode(["when should I submit my work?"]).tolist()

print("no filter :", fcol.query(query_embeddings=q, n_results=1)["documents"][0])
print("academics :", fcol.query(query_embeddings=q, n_results=1, where=___)["documents"][0])
print("food only :", fcol.query(query_embeddings=q, n_results=1, where={"topic": "___"})["documents"][0])

---

## Q8: Cite your sources

Retrieve, answer from the context only, **and print where each fact came from**.

> This is the only task that needs an API key.

In [ ]:
# Hint: results["metadatas"][0] lines up one-to-one with results["documents"][0].
#       For RAG you want the model reading, not inventing - so temperature=0.

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
from litellm import completion

question = "when should I submit my work?"
res = fcol.query(query_embeddings=model.encode([question]).tolist(), n_results=2)

context = "\n\n".join(res["documents"][0])
sources = res["___"][0]

reply = completion(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer using ONLY this context:\n\n" + context},
        {"role": "user", "content": ___},
    ],
    temperature=___,
)

print(reply.choices[0].message.content)
print("\nSources:", [s["topic"] for s in sources])

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **`model.encode()`** | text → 384 numbers, free and local |
| **`cosine_similarity`** | the whole matrix in one call; the outlier has the lowest row average |
| **PCA to 2-D** | a *shadow* of the real space — great for intuition, not for decisions |
| **Naive chunking** | slices words and numbers in half; the fact simply stops existing |
| **`RecursiveCharacterTextSplitter`** | paragraph → sentence → word, so boundaries stay sane |
| **`.tolist()`** | `encode()` returns numpy; Chroma wants plain lists |
| **`where={...}`** | filters *before* the search — a wrong filter forces a wrong answer |
| **Citations** | come from the metadata you stored, never from the model's memory |

**Finished early?**
1. Re-run Q2 with three topics instead of two. Do they still separate cleanly?
2. In Q6, index the same text at `chunk_size=100` and at `chunk_size=800`. Ask the same five questions to each and count how often the right chunk comes back.
3. In Q8, print the `distances` too, and refuse to answer when the best distance is above a cut-off you pick. How did you choose the number?